# Homework 3 — Clustering

> **Study/reference notebook:** The assignment asks learners to write their own code. Use this notebook to study the algorithms, then implement them in the original Coursera notebook. Preserve the original notebook structure, metadata, function signatures, and data paths when submitting.

The exercises implement two iterations of K-means and one Expectation-Maximization update for one-dimensional Gaussian clusters.

## Chapter 3 concepts used here

Clustering is unsupervised: points are grouped without predefined labels. K-means uses hard assignment to the nearest centroid, then updates each centroid to the arithmetic mean of its assigned points. EM uses soft membership: an E-step computes each point's responsibility for each Gaussian component, and an M-step updates each component's mean and standard deviation using those responsibilities as weights.

The assignment gives each cluster a mean and standard deviation but no mixture weights, so this reference implementation assumes equal prior weights. Standard deviations are computed as weighted population standard deviations. Empty K-means clusters retain their previous centroid.

## Assignment rules and imports

Do not install extra packages. Keep the supplied pickle files in `./data/`. The sample data and hidden grader are provided by Coursera and are not included with this notebook.

In [ ]:
import argparse
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from collections import defaultdict
import math

## Problem 1 — K-means clustering (10 points)

Implement exactly two assignment/update iterations for three clusters. At each iteration, assign every point to its nearest current centroid using Euclidean distance, store the assigned points under `cluster1`, `cluster2`, and `cluster3`, then update the centroids to the cluster means. Store the updated centroid list under the corresponding iteration key (`'1'` or `'2'`).

In [ ]:
def k_means_clustering(centroids, dataset):
    result = {
        '1': {'cluster1': [], 'cluster2': [], 'cluster3': [], 'centroids': []},
        '2': {'cluster1': [], 'cluster2': [], 'cluster3': [], 'centroids': []}
    }

    current_centroids = [np.asarray(c, dtype=float) for c in centroids]

    for iteration in range(2):
        cluster_points = [[], [], []]

        for point in dataset:
            point_array = np.asarray(point, dtype=float)
            distances = [np.linalg.norm(point_array - center) for center in current_centroids]
            closest_cluster = int(np.argmin(distances))
            cluster_points[closest_cluster].append(list(point))

        updated_centroids = []
        for index, points in enumerate(cluster_points):
            if points:
                updated_centroids.append(np.mean(np.asarray(points, dtype=float), axis=0).tolist())
            else:
                # Keep the previous center if a cluster receives no points.
                updated_centroids.append(current_centroids[index].tolist())

        iteration_key = str(iteration + 1)
        for index, points in enumerate(cluster_points):
            result[iteration_key][f'cluster{index + 1}'] = points
        result[iteration_key]['centroids'] = updated_centroids
        current_centroids = [np.asarray(c, dtype=float) for c in updated_centroids]

    return result

### K-means validation

The original assignment loads initial centroids, points, and an expected result from three pickle files. Run its visible comparison in Coursera after completing the function. Pay attention to the expected iteration convention: this implementation records the cluster assignments made using the current centers and the updated centers produced from those assignments.

## Problem 2 — EM clustering (10 points)

The dataset is one-dimensional, while each centroid is a `[mean, standard_deviation]` pair. For each point, the E-step evaluates the Gaussian density under every component and normalizes the values into responsibilities. The M-step calculates a weighted mean and weighted population standard deviation for each component. The returned value is a list of updated `[mean, standard_deviation]` pairs after one E/M iteration.

In [ ]:
def em_clustering(centroids, dataset):
    new_centroids = list()
    points = np.asarray(dataset, dtype=float).reshape(-1)
    if len(points) == 0:
        return [[float(mu), float(sigma)] for mu, sigma in centroids]

    means = np.asarray([component[0] for component in centroids], dtype=float)
    stds = np.asarray([abs(component[1]) for component in centroids], dtype=float)
    # Guard against division by zero for a degenerate initial Gaussian.
    safe_stds = np.maximum(stds, 1e-12)

    # E-step: calculate and normalize Gaussian likelihoods for each point.
    likelihoods = np.empty((len(points), len(centroids)), dtype=float)
    for j, (mu, sigma) in enumerate(zip(means, safe_stds)):
        likelihoods[:, j] = (
            np.exp(-0.5 * ((points - mu) / sigma) ** 2)
            / (sigma * math.sqrt(2.0 * math.pi))
        )

    row_sums = likelihoods.sum(axis=1, keepdims=True)
    responsibilities = np.divide(
        likelihoods, row_sums, out=np.full_like(likelihoods, 1.0 / len(centroids)),
        where=row_sums > 0
    )

    # M-step: update each component's weighted mean and standard deviation.
    for j in range(len(centroids)):
        weights = responsibilities[:, j]
        weight_sum = weights.sum()
        if weight_sum <= 0:
            new_centroids.append([float(means[j]), float(stds[j])])
            continue
        new_mean = float(np.sum(weights * points) / weight_sum)
        new_variance = float(np.sum(weights * (points - new_mean) ** 2) / weight_sum)
        new_std = math.sqrt(max(new_variance, 0.0))
        new_centroids.append([new_mean, new_std])

    return new_centroids

### EM validation and assumptions

Run the original visible test using the Coursera-provided pickle files. EM results depend on assumptions about mixture priors and the variance convention. This reference uses equal priors because none are included in the input, and population standard deviation because the M-step is a weighted maximum-likelihood estimate. If the course's visible sample expects a different convention, follow that sample's rubric and output exactly.

## Submission checklist

- Keep the required signatures and return structures unchanged.
- Run the supplied visible tests in the original Coursera environment; the sample pickle files and hidden grader are not available here.
- Submit using the original notebook opened by **Work in browser**, copying code only into its designated answer cells.
- Confirm that each of the two K-means iterations records assignments and updated centroids in the expected order.